In [1]:
# ==========================================================
# Top-10 Products Growth Prediction (2023–2028) with XGBoost
# Table shows: Value (YoY % growth)
# Output: CSV file
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ---------------- 0) Loading dataset ----------------
CSV_PATH = "cleaned_trade_master (1).csv"   # change path if needed
df = pd.read_csv(CSV_PATH)

if "Value_2023" not in df.columns or "Product" not in df.columns:
    raise ValueError("Dataset must have 'Value_2023' and 'Product' columns.")

df["Value_2023"] = pd.to_numeric(df["Value_2023"], errors="coerce")
df = df.dropna(subset=["Value_2023"]).reset_index(drop=True)

# ---------------- 1) Features ----------------
num_feats_all = [
    "Trade_Balance_2023","Growth_2019_2023","Growth_2022_2023",
    "World_Growth","World_Import_Rank","Avg_Distance_km",
    "Concentration","World_Import_Share","Avg_Tariff"
]
cat_feats_all = ["Country","Product","Direction"]

num_cols = [c for c in num_feats_all if c in df.columns]
cat_cols = [c for c in cat_feats_all if c in df.columns]
features = num_cols + cat_cols

X = df[features].copy() if features else pd.DataFrame(index=df.index)
y = df["Value_2023"].astype(float)

# ---------------- 2) XGBoost pipeline ----------------
if features:
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])
    xgb = Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(
            n_estimators=150, max_depth=6, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9, random_state=42, tree_method="hist"
        ))
    ])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    xgb.fit(Xtr, ytr)
    yhat_2023 = xgb.predict(X)
else:
    # fallback: using Value_2023 directly
    yhat_2023 = y.values.copy()

# ---------------- 3) Growth-blend for 2024–2028 ----------------
def to_decimal(s):
    s = pd.to_numeric(s, errors="coerce")
    mask = s.abs() > 2
    s.loc[mask] = s.loc[mask] / 100.0
    return s

g_recent = to_decimal(df["Growth_2022_2023"]) if "Growth_2022_2023" in df else pd.Series(0.0, index=df.index)
g_old    = to_decimal(df["Growth_2019_2023"]) if "Growth_2019_2023" in df else pd.Series(0.0, index=df.index)
g_world  = to_decimal(df["World_Growth"])      if "World_Growth"      in df else pd.Series(0.0, index=df.index)

g_blend  = (0.6*g_recent.fillna(0.0) +
            0.4*(g_old.fillna(0.0)/4.0) +
            0.3*g_world.fillna(0.0)).clip(-0.5, 1.5)

years = np.array([2023, 2024, 2025, 2026, 2027, 2028])
k = np.arange(len(years))
multipliers = np.power(1.0 + g_blend.values.reshape(-1, 1), k.reshape(1, -1))
pred_matrix = yhat_2023.reshape(-1, 1) * multipliers

# ---------------- 4) Aggregating Product × Year ----------------
pred_long = (
    pd.DataFrame(pred_matrix, columns=years)
      .assign(Product=df["Product"].astype(str).values)
      .melt(id_vars=["Product"], var_name="Year", value_name="Projected_Value")
)
prod_year = (pred_long.groupby(["Product","Year"], as_index=False)["Projected_Value"].sum()
                       .sort_values(["Product","Year"]))
prod_year["YoY_%"] = prod_year.groupby("Product")["Projected_Value"].pct_change() * 100.0

# ---------------- 5) Ranking by 5-year CAGR ----------------
p2023 = prod_year[prod_year["Year"] == 2023][["Product","Projected_Value"]].rename(columns={"Projected_Value":"V2023"})
p2028 = prod_year[prod_year["Year"] == 2028][["Product","Projected_Value"]].rename(columns={"Projected_Value":"V2028"})
summary = p2023.merge(p2028, on="Product", how="inner")
summary["CAGR_%"] = np.where(
    summary["V2023"] <= 0, np.nan,
    (np.power(summary["V2028"]/summary["V2023"], 1/5) - 1.0) * 100.0
)

# filtering out tiny baselines (optional)
median_base = summary["V2023"].replace(0, np.nan).median()
summary_f = summary[summary["V2023"] >= median_base]

top_products = (summary_f.sort_values("CAGR_%", ascending=False)
                           .head(10)["Product"].tolist())

# ---------------- 6) Formatting output table ----------------
sub = prod_year[(prod_year["Product"].isin(top_products)) & (prod_year["Year"] >= 2023)].copy()

def fmt_val(v): 
    return f"{v:,.0f}"
def fmt_pct(p):
    return "" if pd.isna(p) else f"{p:.1f}%"

sub["Display"] = np.where(
    sub["Year"].eq(2023),
    sub["Projected_Value"].map(fmt_val),
    sub.apply(lambda r: f"{fmt_val(r['Projected_Value'])} ({fmt_pct(r['YoY_%'])})", axis=1)
)

wide = (sub.pivot(index="Product", columns="Year", values="Display")
          .reindex(columns=years)
          .loc[top_products])

# ---------------- 7) Saving to CSV ----------------
OUT_CSV = "top10_products_growth_2023_2028.csv"
wide.to_csv(OUT_CSV)

print(f"Saved to {OUT_CSV}")
print(wide)


C:\Users\chaud\AppData\Local\Temp\ipykernel_18020\2672045341.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.19  0.07  0.08 ... -0.17  0.11  0.1 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  s.loc[mask] = s.loc[mask] / 100.0


Saved to top10_products_growth_2023_2028.csv
Year                                                      2023  \
Product                                                          
Copper and articles thereof                         11,091,245   
Aircraft, spacecraft, and parts thereof             10,228,761   
Articles of apparel and clothing accessories, k...   9,615,750   
Live animals                                         4,737,433   
Dairy produce; birds' eggs; natural honey; edib...  12,400,230   
Tobacco and manufactured tobacco substitutes; p...   6,067,730   
Footwear, gaiters and the like; parts of such a...   6,535,200   
Soap, organic surface-active agents, washing pr...   4,665,955   
Ships, boats and floating structures                 9,717,804   
Preparations of cereals, flour, starch or milk;...   7,600,477   

Year                                                              2024  \
Product                                                                  
Copper and art